In [15]:
!pip install reportlab
import gradio as gr
from datetime import datetime
from pathlib import Path
import json
import re
import ast
from reportlab.lib.pagesizes import letter, A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak, Table, TableStyle, Preformatted
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
import requests
import urllib.parse
import inspect

class DocGenieAnalyzer:
    """Analyze Python functions and generate professional docstrings"""

    @staticmethod
    def extract_function_signature(code):
        """Extract function name, parameters, return type from code"""
        try:
            tree = ast.parse(code)
            func_def = tree.body[0]
            if not isinstance(func_def, ast.FunctionDef):
                return None, None

            func_name = func_def.name
            params = []

            for arg in func_def.args.args:
                param_type = "Any"sSSS
                if arg.annotation:
                    param_type = ast.unparse(arg.annotation) if hasattr(ast, 'unparse') else "Any"
                params.append({
                    'name': arg.arg,
                    'type': param_type,
                    'default': None
                })
            for i, default in enumerate(func_def.args.defaults):
                idx = len(params) - len(func_def.args.defaults) + i
                if idx >= 0:
                    params[idx]['default'] = ast.unparse(default) if hasattr(ast, 'unparse') else str(default)

            return_type = "Any"
            if func_def.returns:
                return_type = ast.unparse(func_def.returns) if hasattr(ast, 'unparse') else "Any"

            return {
                'name': func_name,
                'params': params,
                'return_type': return_type
            }, func_def
        except:
            return None, None

    @staticmethod
    def analyze_function_logic(func_def, code):
        """Deep analysis of function body to understand actual behavior"""
        try:
            body = func_def.body
            analysis = {
                'has_loops': False,
                'has_conditions': False,
                'has_return': False,
                'loop_types': [],
                'condition_types': [],
                'operations': [],
                'return_types': [],
                'variables_used': set(),
                'function_calls': [],
                'description': ""
            }

            # Walk through AST
            for node in ast.walk(func_def):
                if isinstance(node, ast.For):
                    analysis['has_loops'] = True
                    analysis['loop_types'].append('for')
                elif isinstance(node, ast.While):
                    analysis['has_loops'] = True
                    analysis['loop_types'].append('while')
                elif isinstance(node, ast.If):
                    analysis['has_conditions'] = True
                    analysis['condition_types'].append('if')
                elif isinstance(node, ast.Return):
                    analysis['has_return'] = True
                    if node.value:S
                        analysis['return_types'].append(ast.unparse(node.value) if hasattr(ast, 'unparse') else 'value')
                elif isinstance(node, ast.BinOp):
                    op = node.op
                    if isinstance(op, ast.Mult):
                        analysis['operations'].append('multiplication')
                    elif isinstance(op, ast.Add):
                        analysis['operations'].append('addition')
                    elif isinstance(op, ast.Sub):
                        analysis['operations'].append('subtraction')
                    elif isinstance(op, ast.Div):
                        analysis['operations'].append('division')
                elif isinstance(node, ast.Call):
                    if isinstance(node.func, ast.Name):
                        analysis['function_calls'].append(node.func.id)
                elif isinstance(node, ast.Name):
                    analysis['variables_used'].add(node.id)

            # Build detailed description
            description_parts = []

            if func_def.name:
                description_parts.append(f"Executes {func_def.name} operation")

            if analysis['has_conditions']:
                description_parts.append("with conditional logic")

            if analysis['has_loops']:
                description_parts.append("and iterates through values")

            if 'multiplication' in analysis['operations'] and 'subtraction' in analysis['operations']:
                description_parts.append("performing mathematical calculations")
            elif 'multiplication' in analysis['operations']:
                description_parts.append("using multiplication operations")
            elif 'addition' in analysis['operations']:
                description_parts.append("using addition operations")
            elif 'division' in analysis['operations']:
                description_parts.append("using division operations")

            if analysis['return_types']:
                description_parts.append(f"and returns the computed result")

            analysis['description'] = " ".join(description_parts) + "."
            if len(analysis['description']) < 20:
                analysis['description'] = "Performs computation based on input parameters."

            return analysis
        except Exception as e:
            return {
                'has_loops': False,
                'has_conditions': False,
                'has_return': False,
                'description': "Executes function logic based on provided parameters."
            }

    @staticmethod
    def generate_google_docstring(signature, analysis, code):
        """Generate accurate Google-style docstring"""
        summary = analysis.get('description', 'Processes input and returns result.')

        docstring = f' """{summary}\n'

        if signature['params']:
            docstring += '\n Args:\n'
            for param in signature['params']:
                default_str = f" (default: {param['default']})" if param['default'] else ""
                docstring += f" {param['name']} ({param['type']}): Parameter for controlling {param['name']} behavior{default_str}.\n"

        docstring += f"\n Returns:\n {signature['return_type']}: The result of the computation.\n"
        docstring += ' """\n'

        return docstring

    @staticmethod
    def generate_numpy_docstring(signature, analysis, code):
        """Generate accurate NumPy-style docstring"""
        summary = analysis.get('description', 'Processes input and returns result.')

        docstring = f' """{summary}\n'

        if signature['params']:
            docstring += '\n Parameters\n --------\n'
            for param in signature['params']:
                default_str = f", default: {param['default']}" if param['default'] else ""
                docstring += f" {param['name']} : {param['type']}{default_str}\n"
                docstring += f" Parameter for controlling {param['name']} behavior.\n"

        docstring += f"\n Returns\n -----\n {signature['return_type']}\n"
        docstring += f" The result of the computation.\n"
        docstring += ' """\n'

        return docstring

analyzer = DocGenieAnalyzer()
generation_history = []

SyntaxError: invalid syntax (1004766957.py, line 33)

In [16]:
# File Upload
with gr.Blocks() as demo:
    code_input = gr.Code(
        language="python",
        label="Input Function Code",
        lines=15,
        value="""def example_function(param1: str, param2: int = 10) -> bool:
    '''This is a sample function for demonstration.'''
    if len(param1) > param2:
        for i in range(param2):
            print(f"Processing {param1[i]}")
        return True
    return False
""",
        interactive=True
    )
    file_upload = gr.File(label="📤 Or Upload .py File", file_types=['.py'])
    upload_status = gr.Textbox(label="Upload Status", interactive=False, lines=1)

    def process_and_display(file):
        if file:
            with open(file.name, 'r') as f:
                content = f.read()
            return content, "File uploaded successfully!"
        return "", "No file selected."

    file_upload.change(process_and_display, inputs=[file_upload], outputs=[code_input, upload_status])

    # Style Selection
    with gr.Row():
        docstring_style = gr.Radio(["google", "numpy"], value="google", label="🎨 Docstring Style")
        generate_btn = gr.Button("✨ Generate Docstring", size="lg", variant="primary", scale=2)

    # Output Area
    gr.Markdown("### 📄 Generated Output")

    code_output = gr.Code(
        language="python",
        label="Function with Docstring",
        lines=15,
        interactive=False
    )
    gen_status = gr.Textbox(label="Status", interactive=False, lines=1)

    def generate_docstring(code, style):
        if not code:
            return "", "Please provide some code to generate a docstring."
        try:
            signature, func_def = analyzer.extract_function_signature(code)
            if not signature:
                return "", "Could not extract function signature. Please check your code."

            analysis = analyzer.analyze_function_logic(func_def, code)

            if style == "google":
                docstring = analyzer.generate_google_docstring(signature, analysis, code)
            elif style == "numpy":
                docstring = analyzer.generate_numpy_docstring(signature, analysis, code)
            else:
                docstring = ""

            # Insert docstring into code
            lines = code.splitlines()
            if func_def:
                # Find the line number of the function definition
                # Adjust for decorators and 'def' line
                start_line_num = func_def.lineno
                # Find where to insert the docstring (after the def line and any decorators)
                insertion_idx = start_line_num
                # Find the indentation level of the function definition
                indentation = len(lines[start_line_num - 1]) - len(lines[start_line_num - 1].lstrip())

                # Add appropriate indentation to the docstring
                indented_docstring = " " * (indentation + 4) + docstring.replace('\n', '\n' + ' ' * (indentation + 4)).strip() + '\n'

                # Reconstruct the code
                new_code_lines = lines[:insertion_idx]
                new_code_lines.append(indented_docstring)
                new_code_lines.extend(lines[insertion_idx:])
                final_code = "\n".join(new_code_lines)
            else:
                final_code = code # Fallback if signature extraction fails

            generation_history.append({
                'code': code,
                'style': style,
                'docstring': docstring,
                'final_code': final_code,
                'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            })
            return final_code, "Docstring generated successfully!"
        except Exception as e:
            return "", f"Error generating docstring: {e}"

    generate_btn.click(
        generate_docstring,
        inputs=[code_input, docstring_style],
        outputs=[code_output, gen_status]
    )

    with gr.Column(scale=1):
        gr.Markdown("### 🎮 Controls")

        clear_btn = gr.Button("🗑️ Clear History", variant="stop", size="sm")
        def clear_history():
            global generation_history
            generation_history = []
            return "History Cleared!"
        clear_btn.click(clear_history, outputs=[upload_status])

        gr.Markdown("---")
        gr.Markdown("### 📥 Export")

        pdf_btn = gr.Button("📄 Download PDF", size="sm", variant="primary")
        txt_btn = gr.Button("📋 Download TXT", size="sm", variant="primary")

        pdf_file = gr.File(label="PDF File", visible=False)
        txt_file = gr.File(label="TXT File", visible=False)
        export_status = gr.Textbox(label="Export Status", interactive=False, lines=2)

        def download_pdf():
            if not generation_history:
                return "No history to export!", None

            doc = SimpleDocTemplate("docstring_report.pdf", pagesize=letter)
            styles = getSampleStyleSheet()
            elements = []

            elements.append(Paragraph("Docstring Generation Report", styles['h1']))
            elements.append(Spacer(1, 0.2 * inch))
            elements.append(Paragraph(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", styles['Normal']))
            elements.append(Spacer(1, 0.2 * inch))

            for entry in generation_history:
                elements.append(Paragraph(f"Function Code (Style: {entry['style'].capitalize()}):", styles['h2']))
                elements.append(Preformatted(entry['code'], styles['Code']))
                elements.append(Spacer(1, 0.1 * inch))
                elements.append(Paragraph("Generated Docstring:", styles['h3']))
                elements.append(Preformatted(entry['docstring'], styles['Code']))
                elements.append(Spacer(1, 0.1 * inch))
                elements.append(Paragraph("Final Code:", styles['h3']))
                elements.append(Preformatted(entry['final_code'], styles['Code']))
                elements.append(Spacer(1, 0.4 * inch))
                elements.append(PageBreak())

            try:
                doc.build(elements)
                return "PDF generated successfully!", "docstring_report.pdf"
            except Exception as e:
                return f"Error generating PDF: {e}", None

        pdf_btn.click(download_pdf, outputs=[export_status, pdf_file])

        def download_txt():
            if not generation_history:
                return "No history to export!", None

            txt_content = []
            for entry in generation_history:
                txt_content.append(f"--- Docstring Generated on {entry['timestamp']} ---")
                txt_content.append(f"Style: {entry['style'].capitalize()}")
                txt_content.append("\nOriginal Code:\n")
                txt_content.append(entry['code'])
                txt_content.append("\nGenerated Docstring:\n")
                txt_content.append(entry['docstring'])
                txt_content.append("\nFinal Code:\n")
                txt_content.append(entry['final_code'])
                txt_content.append("\n")

            with open("docstring_report.txt", "w") as f:
                f.write("\n".join(txt_content))

            return "TXT generated successfully!", "docstring_report.txt"

        txt_btn.click(download_txt, outputs=[export_status, txt_file])

        gr.Markdown("---")
        gr.Markdown("### 📱 Share")

        whatsapp_btn = gr.Button("💬 WhatsApp", size="sm", variant="secondary")
        facebook_btn = gr.Button("👍 Facebook", size="sm", variant="secondary")

        share_output = gr.Textbox(label="Share Link", interactive=False, lines=4)

        def generate_whatsapp_link():
            last_entry = generation_history[-1] if generation_history else None
            if last_entry:
                message = f"Check out this Python docstring I generated with Doc-Genie!\n\nCode:\n{last_entry['final_code']}\n\n#DocGenie #Python #Docstrings"
                encoded_message = urllib.parse.quote(message)
                return f"https://api.whatsapp.com/send?text={encoded_message}"
            return "No docstring generated yet to share."

        def generate_facebook_link():
            last_entry = generation_history[-1] if generation_history else None
            if last_entry:
                # Facebook sharing is typically done via a URL with content or a share dialog
                # For simplicity, we'll generate a link that encourages sharing the code manually.
                message = f"I just generated a professional docstring for my Python code using Doc-Genie!\n\n{last_entry['final_code']}\n\n#Python #Docstrings #AI"
                encoded_message = urllib.parse.quote(message)
                return f"https://www.facebook.com/sharer/sharer.php?quote={encoded_message}" # This is a basic share link
            return "No docstring generated yet to share."

        whatsapp_btn.click(generate_whatsapp_link, outputs=[share_output])
        facebook_btn.click(generate_facebook_link, outputs=[share_output])

        gr.Markdown("---")
        gr.Markdown("""
### 📊 Features

*Improvements:*
- ✅ Accurate logic analysis
- ✅ Contextual descriptions
- ✅ Operation detection
- ✅ Loop & condition recognition
- ✅ Better PDF export
- ✅ Consistent output
### 🎯 Supported Styles

*Google:* Standard format
*NumPy:* Scientific format
""")

print("\n✅ Doc-Genie v2.0 Ready!\n")
demo.launch(share=True, show_error=True, show_api=False)

/tmp/ipykernel_491/2324627861.py:224: DeprecationWarning: The 'show_api' parameter in launch() will be removed in Gradio 6.0. You will need to use the 'footer_links' parameter instead. To replicate show_api=False, In Gradio 6.0, use footer_links=['gradio', 'settings'].
  demo.launch(share=True, show_error=True, show_api=False)



✅ Doc-Genie v2.0 Ready!

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dd0763d1404a24b549.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
